# Traduction multilingue *many-to-many* (NLLB + LoRA) — éwé · anglais · français

Un **seul adaptateur** LoRA pour plusieurs **directions** de traduction, au lieu d'un modèle par couple.

**Idée clé (NLLB) :** la langue de sortie est choisie par un *token de langue* placé au début des `labels`
(et, à l'inférence, via `forced_bos_token_id`). Il suffit de mélanger, dans le même corpus, des exemples
de plusieurs directions, chacun avec ses `labels` préfixés du token de la langue cible :

- `ewe_Latn → eng_Latn`
- `ewe_Latn → fra_Latn`
- `eng_Latn → fra_Latn`
- `fra_Latn → eng_Latn`

On obtient un modèle compact qui partage ses connaissances entre directions (utile pour l'éwé, peu doté,
et pour le couple anglais↔français ajouté au dataset).

In [ ]:
!pip install -q evaluate sacrebleu

In [ ]:
import json
import os
import re
from pathlib import Path

# Un seul GPU (Kaggle T4 x2 -> DataParallel sature la VRAM sinon)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import numpy as np
import evaluate
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"

## 0. Données & sécurité

Par défaut on charge le dataset **local** `data/processed/translation/` (mis à jour par
`add_fr_en_to_nllb.py` : éwé + anglais/français). Pour un entraînement Kaggle/GPU,
utilisez la variante Hub commentée ci-dessous.

> ⚠️ **Sécurité** : ne jamais coder un token Hugging Face en dur dans le notebook.
> Utilisez `huggingface-cli login` ou la variable d'environnement `HF_TOKEN`.
> Le token précédemment exposé dans `traduction_multilingue.ipynb` doit être **révoqué**.

In [ ]:
# --- Authentification Hugging Face (tokens JAMAIS en dur) ----------------
# Kaggle : Add-ons -> Secrets -> creer 2 secrets : HF_TOKEN_READ et HF_TOKEN_WRITE
#          (tokens NEUFS, l'ancien ayant ete revoque), puis les attacher au notebook.
# Local  : un fichier .env (non versionne) avec HF_TOKEN_READ / HF_TOKEN_WRITE,
#          ou `huggingface-cli login`.
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN_READ  = _get_secret("HF_TOKEN_READ")    # acces aux datasets prives
HF_TOKEN_WRITE = _get_secret("HF_TOKEN_WRITE")   # push de l'adaptateur

# Depots Hugging Face -- NOUVEAUX noms pour NE PAS ecraser les anciens.
HF_USERNAME      = "romaricnadjire"
HUB_DATASET_REPO = f"{HF_USERNAME}/ewe-en-fr-nllb-translation"        # ancien: ewe-nllb-translation
HUB_ADAPTER_REPO = f"{HF_USERNAME}/nllb-ewe-en-fr-multilingual-lora"  # ancien: nllb-ewe-multilingual-lora
STAGE1_ADAPTER_REPO = f"{HF_USERNAME}/nllb-ewe-stage1-mined-lora"  # CASCADE : point de depart du Stage-2
PUSH_PRIVATE     = True

# Donnees : local si present (machine perso), sinon depuis le Hub (Kaggle).
DATA_DIR = "./data/processed/translation"
SPLITS = {"train": "train.jsonl", "validation": "validation.jsonl", "test": "test.jsonl"}

# Inclure les paires MINEES (ewe-en / ewe-fr filtrees par similarite) dans le train.
# Mettre a False pour n'entrainer que sur les sources propres (train.jsonl seul).
USE_MINED  = False
MINED_FILE = "train_mined.jsonl"

# Schema EXPLICITE : sinon le loader JSON deduit le schema du 1er bloc (train_mined
# commence par des paires ewe-en) puis echoue sur les paires ewe-fr suivantes.
from datasets import Features, Value
TRANS_FEATURES = Features({"translation": {
    "ewe_Latn": Value("string"),
    "eng_Latn": Value("string"),
    "fra_Latn": Value("string"),
}})

def _load_splits(files):
    if os.path.isdir(DATA_DIR):
        return load_dataset("json", data_files={k: f"{DATA_DIR}/{v}" for k, v in files.items()},
                            features=TRANS_FEATURES)
    return load_dataset(HUB_DATASET_REPO, data_files=files, features=TRANS_FEATURES,
                        token=HF_TOKEN_READ or True)

print(f"Chargement depuis {DATA_DIR if os.path.isdir(DATA_DIR) else 'Hub : ' + HUB_DATASET_REPO}")
ds_raw = _load_splits(SPLITS)

if USE_MINED:
    try:
        mined = _load_splits({"train": MINED_FILE})["train"]
        before = len(ds_raw["train"])
        ds_raw["train"] = concatenate_datasets([ds_raw["train"], mined])
        print(f"  + {MINED_FILE} : {len(mined)} paires minees "
              f"(train {before} -> {len(ds_raw['train'])})")
    except Exception as e:
        print(f"  {MINED_FILE} ignore ({type(e).__name__}: {e})")
ds_raw


## 1. Configuration

`DIRECTIONS` contient la **liste des couples (source, cible)** entraînés par le même
adaptateur. Pour en ajouter un (ex. `("ewe_Latn", "yor_Latn")`), il suffit de l'ajouter ici,
à condition que les deux clés existent dans le dataset.

In [ ]:
MODEL_NAME   = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR   = "./output/nllb-multi-mt"
ADAPTER_DIR  = "./output/nllb-multi-mt/adapter"
RESULTS_FILE = "./output/resultats_multi_mt.json"

# Directions (src, tgt) entraînées par le MÊME adaptateur LoRA.
DIRECTIONS = [
    ("ewe_Latn", "eng_Latn"),
    ("eng_Latn", "ewe_Latn"),
    ("ewe_Latn", "fra_Latn"),
    ("fra_Latn", "ewe_Latn"),
    ("eng_Latn", "fra_Latn"),
    ("fra_Latn", "eng_Latn"),
]

# Equilibrage des directions : plafond d'exemples d'ENTRAINEMENT par direction (None = aucun).
# La symetrie est deja assuree par les directions inverses ci-dessus ; ce plafond evite que
# eng<->fra (~168k/dir) ne noie les directions vers l'ewe (~63-75k/dir). 80000 garde 100% de l'ewe.
MAX_PAIRS_PER_DIR = 80000

MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

LEARNING_RATE    = 3e-4
BATCH_SIZE_TRAIN = 4
BATCH_SIZE_EVAL  = 8
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01

LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration OK")
print(f"  Directions     : {DIRECTIONS}")
print(f"  Batch effectif : {BATCH_SIZE_TRAIN} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE_TRAIN * GRAD_ACCUM_STEPS}")
print(f"  Adaptateur Hub : {HUB_ADAPTER_REPO}")


## 2. Nettoyage (par direction)

`clean_pair` valide **une** paire (source, cible). On l'applique séparément pour chaque
direction. On retire : les paires vides, les copies (source == cible), les références
bibliques seules, et les caractères éwé qui « fuiteraient » dans une cible non-éwé.

In [ ]:
EWE_CHARS = set("ŋɖɔɛʋƒãẽĩõũ")
BIBLE_REF_RE = re.compile(r"^\s*\d{1,3}:\d{1,3}(?:-\d{1,3})?\s*$")

def clean_pair(src, tgt, tgt_lang):
    # Retourne True si la paire (src, tgt) est exploitable.
    src = (src or "").strip()
    tgt = (tgt or "").strip()
    if not src or not tgt:
        return False
    if src == tgt:                       # copie, pas une traduction
        return False
    if BIBLE_REF_RE.match(src) and src == tgt:
        return False
    if tgt_lang != "ewe_Latn" and sum(c in EWE_CHARS for c in tgt) >= 2:
        return False
    return True

# Apercu : combien de paires propres par direction ?
for src_lang, tgt_lang in DIRECTIONS:
    n = sum(
        clean_pair(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang), tgt_lang)
        for ex in ds_raw["train"]
    )
    print(f"  train {src_lang} -> {tgt_lang} : {n} paires propres")

## 3. Tokenisation multilingue

Le coeur du notebook. Pour **chaque** direction `(src, tgt)` :

1. on règle `tokenizer.src_lang` et `tokenizer.tgt_lang` ;
2. `text_target=` fait préfixer automatiquement les `labels` avec le token de la langue
   cible — c'est ce token qui dit au modèle quelle langue produire ;
3. on concatène les sous-ensembles tokenisés, puis on **mélange** (`shuffle`) pour que
   chaque batch contienne un mix des directions.

Chaque ligne du dataset n'a que les clés présentes : un exemple éwé↔anglais n'a pas de
`fra_Latn`, donc le filtre `clean_pair` ne le retient que pour les directions pertinentes.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_preprocess(src_lang, tgt_lang):
    def preprocess(batch):
        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang          # -> token de langue ajoute aux labels
        sources = [ex.get(src_lang) or "" for ex in batch["translation"]]
        targets = [ex.get(tgt_lang) or "" for ex in batch["translation"]]
        model_inputs = tokenizer(
            sources, text_target=targets, max_length=MAX_INPUT_LEN, truncation=True
        )
        model_inputs["labels"] = [ids[:MAX_TARGET_LEN] for ids in model_inputs["labels"]]
        return model_inputs
    return preprocess

parts = {"train": [], "validation": [], "test": []}
for src_lang, tgt_lang in DIRECTIONS:
    sub = ds_raw.filter(
        lambda ex, s=src_lang, t=tgt_lang: clean_pair(
            ex["translation"].get(s), ex["translation"].get(t), t
        )
    )
    # Equilibrage : plafonne le TRAIN par direction (eng<->fra ne doit pas dominer l'ewe).
    if MAX_PAIRS_PER_DIR and len(sub["train"]) > MAX_PAIRS_PER_DIR:
        sub["train"] = sub["train"].shuffle(seed=42).select(range(MAX_PAIRS_PER_DIR))
    print(f"  {src_lang}->{tgt_lang} : train={len(sub['train'])} paires propres")
    tok = sub.map(
        make_preprocess(src_lang, tgt_lang),
        batched=True,
        remove_columns=ds_raw["train"].column_names,
        desc=f"Tokenisation {src_lang}->{tgt_lang}",
    )
    for split in parts:
        parts[split].append(tok[split])

tokenized = DatasetDict({
    split: concatenate_datasets(p).shuffle(seed=42) for split, p in parts.items()
})
print(tokenized)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

## 4. LoRA + modèle

On gèle NLLB et on n'entraîne que les petites matrices LoRA sur `q_proj` / `v_proj`.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Cascade Stage-2 : on REPART de l'adaptateur Stage-1 (mines realignees) au lieu d'un LoRA neuf.
# is_trainable=True -> les poids LoRA restent entrainables pour affiner sur le PROPRE.
# Aucun conflit de dimensions : les tokens de langue sont dans le modele de base (gele),
# le LoRA (q_proj/v_proj) est independant de la direction -> structure identique au Stage-1.
model = PeftModel.from_pretrained(model, STAGE1_ADAPTER_REPO, is_trainable=True,
                                  token=HF_TOKEN_READ)
model.enable_input_require_grads()   # requis : gradient_checkpointing + PEFT (base gelee)
model.print_trainable_parameters()   # doit afficher 2,359,296 (= Stage-1) -> chainage OK

## 5. Entraînement

Le jeu de validation mélange les 4 directions. Comme `predict_with_generate` ne peut forcer
qu'**une** langue de sortie à la fois, un BLEU « mélangé » n'aurait pas de sens : on suit la
**perte de validation** (`eval_loss`) pendant l'entraînement, puis on calcule BLEU / chrF++
**séparément par direction** à la fin (section 6).

In [ ]:
eval_subset = tokenized["validation"].select(range(min(800, len(tokenized["validation"]))))

training_args = Seq2SeqTrainingArguments(
    output_dir = OUTPUT_DIR,
    num_train_epochs = NUM_EPOCHS,
    eval_steps    = 1000,
    save_steps    = 1000,
    logging_steps = 50,
    per_device_train_batch_size = BATCH_SIZE_TRAIN,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    gradient_checkpointing = True,
    learning_rate     = LEARNING_RATE,
    warmup_ratio      = WARMUP_RATIO,
    lr_scheduler_type = "cosine",
    weight_decay      = WEIGHT_DECAY,
    fp16 = (device == "cuda"),
    # Pas de generation pendant l'eval (cibles melangees) -> on suit eval_loss
    predict_with_generate = False,
    eval_strategy          = "steps",
    save_strategy          = "steps",
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_loss",
    greater_is_better      = False,
    save_total_limit       = 2,
    disable_tqdm           = True,
    logging_first_step     = True,
    report_to              = "none",
    # Checkpoints pousses sur le Hub pour REPRENDRE entre sessions Kaggle (12h max).
    # Seuls les parametres LoRA ont un etat optimiseur -> checkpoints legers.
    push_to_hub      = bool(HF_TOKEN_WRITE),
    hub_model_id     = HUB_ADAPTER_REPO,
    hub_strategy     = "checkpoint",
    hub_private_repo = PUSH_PRIVATE,
    hub_token        = HF_TOKEN_WRITE,
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized["train"],
    eval_dataset     = eval_subset,
    processing_class = tokenizer,
    data_collator    = data_collator,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=3)],
)
print("Trainer multilingue pret.")

In [ ]:
# Reprise multi-session. Sur Kaggle, /kaggle/working est efface entre sessions :
# on restaure le dernier checkpoint depuis le Hub (cf. hub_strategy="checkpoint").
from huggingface_hub import snapshot_download

output_path = Path(OUTPUT_DIR)

def _last_local_ckpt():
    if not output_path.is_dir():
        return None
    ckpts = sorted(
        [d for d in output_path.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    return str(ckpts[-1]) if ckpts else None

last_ckpt = _last_local_ckpt()

# Pas de checkpoint local (nouvelle session Kaggle) -> tenter de le recuperer sur le Hub.
if last_ckpt is None and HF_TOKEN_WRITE:
    try:
        snapshot_download(
            repo_id=HUB_ADAPTER_REPO,
            allow_patterns="last-checkpoint/*",
            local_dir=OUTPUT_DIR,
            token=HF_TOKEN_WRITE,
        )
        cand = output_path / "last-checkpoint"
        if (cand / "trainer_state.json").exists():
            last_ckpt = str(cand)
            print(f"Checkpoint restaure depuis le Hub : {last_ckpt}")
    except Exception as e:
        print(f"Aucun checkpoint Hub a restaurer ({type(e).__name__}: {e})")

print(f"Reprise depuis : {last_ckpt}" if last_ckpt else "Entrainement from scratch.")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\nAdaptateur multilingue sauvegarde : {ADAPTER_DIR}")
print(f"Loss train finale : {train_result.training_loss:.4f}")

## 6. Évaluation finale — par direction

On recharge le meilleur modèle (LoRA fusionné), puis on évalue **chaque** direction
séparément en forçant la bonne langue de sortie via `forced_bos_token_id`.

In [ ]:
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric      = evaluate.load("chrf")

print("Chargement du modele fine-tune...")
base_model_ft = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_ft = PeftModel.from_pretrained(base_model_ft, ADAPTER_DIR).merge_and_unload().to(device)
model_ft.eval()
tokenizer_ft = AutoTokenizer.from_pretrained(ADAPTER_DIR)

def translate_batch(model, tok, sources, src_lang, tgt_lang, batch_size=16, max_new_tokens=128):
    tok.src_lang = src_lang
    forced_bos = tok.convert_tokens_to_ids(tgt_lang)
    preds = []
    for i in range(0, len(sources), batch_size):
        batch = sources[i:i + batch_size]
        inputs = tok(batch, return_tensors="pt", padding=True, truncation=True,
                     max_length=MAX_INPUT_LEN).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                 max_new_tokens=max_new_tokens, num_beams=4)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

def eval_direction(split, src_lang, tgt_lang, n=None):
    pairs = [
        (ex["translation"].get(src_lang), ex["translation"].get(tgt_lang))
        for ex in split
        if clean_pair(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang), tgt_lang)
    ]
    if n:
        pairs = pairs[:n]
    sources = [p[0] for p in pairs]
    refs    = [p[1] for p in pairs]
    preds   = translate_batch(model_ft, tokenizer_ft, sources, src_lang, tgt_lang)
    bleu = sacrebleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    chrf = chrf_metric.compute(predictions=preds, references=[[r] for r in refs], word_order=2)
    return {"n": len(sources), "bleu": round(bleu["score"], 2), "chrf++": round(chrf["score"], 2)}

# n=None -> tout le test. Mettre une valeur (ex. 500) pour un apercu rapide.
EVAL_N = None

results = {}
for src_lang, tgt_lang in DIRECTIONS:
    key = f"{src_lang}->{tgt_lang}"
    print(f"=== test : {key} ===")
    results[key] = eval_direction(ds_raw["test"], src_lang, tgt_lang, n=EVAL_N)
    print(f"  BLEU={results[key]['bleu']}  chrF++={results[key]['chrf++']}  (n={results[key]['n']})")

print("\n===== Récapitulatif =====")
print(f"{'direction':<22}{'n':>8}{'BLEU':>8}{'chrF++':>9}")
for key, r in results.items():
    print(f"{key:<22}{r['n']:>8}{r['bleu']:>8}{r['chrf++']:>9}")

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\nResultats sauvegardes : {RESULTS_FILE}")

In [ ]:
# Exemples : quelques phrases traduites dans chaque direction
for src_lang, tgt_lang in DIRECTIONS:
    pairs = [
        ex["translation"] for ex in ds_raw["test"]
        if clean_pair(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang), tgt_lang)
    ][:4]
    sources = [t[src_lang] for t in pairs]
    preds = translate_batch(model_ft, tokenizer_ft, sources, src_lang, tgt_lang)
    print(f"\n----- {src_lang} -> {tgt_lang} -----")
    for s, p in zip(sources, preds):
        print(f"  SRC : {s}")
        print(f"  ->  : {p}\n")

## 7. Publication de l'adaptateur (nouveau nom)

On pousse l'adaptateur LoRA et les résultats vers un **nouveau dépôt**
(`HUB_ADAPTER_REPO`), ce qui **préserve l'ancien** `nllb-ewe-multilingual-lora`.

> Le token doit être chargé (Secrets Kaggle ou `huggingface-cli login`) — voir la
> cellule de la section 0. **Aucun token n'est écrit en dur.**


In [ ]:
from huggingface_hub import HfApi, login

if HF_TOKEN_WRITE:
    login(token=HF_TOKEN_WRITE, add_to_git_credential=False)
else:
    print("ATTENTION : HF_TOKEN_WRITE absent — ajoutez-le dans les Secrets Kaggle pour pousser.")

api = HfApi()
api.create_repo(repo_id=HUB_ADAPTER_REPO, repo_type="model",
                private=PUSH_PRIVATE, exist_ok=True)

# Adaptateur LoRA + tokenizer
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=HUB_ADAPTER_REPO,
    repo_type="model",
    commit_message="Adaptateur LoRA many-to-many (ewe/en/fr) - 4 directions",
)

# Resultats d'evaluation (si presents)
if os.path.exists(RESULTS_FILE):
    api.upload_file(
        path_or_fileobj=RESULTS_FILE,
        path_in_repo="resultats_multi_mt.json",
        repo_id=HUB_ADAPTER_REPO,
        repo_type="model",
        commit_message="Resultats BLEU/chrF++ par direction",
    )

print(f"Adaptateur publie : https://huggingface.co/{HUB_ADAPTER_REPO}")
